# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ssaqlain09/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason code

*Write the rule in plain words first. Then the reason code it can output.*

**Lane (confirmed):** Refresh / Content Opportunity Scoring — I am keeping the lane I picked in Week 1.

**The rule in plain words.** *"A page is worth a reviewer's time first if people see it a lot, but far fewer
click it than usual for where it ranks — and it has not been updated for a while."* CTR-vs-position is the
main ingredient, staleness is a small extra, and pages with little visibility are not scored at all.

**One reason code:** `low_ctr_for_position` — the page is visible (500+ impressions in 90 days, a real
average position) and its CTR is under half the median CTR of pages in the same position tier.
**Action label:** `review_title_snippet` when the code fires, otherwise `monitor`.

**Rule inputs (all observed, none derived from the label):** `impressions_90d`, `avg_position`, `ctr`,
`position_tier`, `days_since_last_update`. The 30-day trend columns are off limits — they *define* the proxy
label (`trend_direction == "down"`), which I use only to check signals and to score the rule afterwards.

Both signals below are checked *before* the rule is written, on the 30,000-row starter data.

In [1]:
import os, sys, subprocess, json
import numpy as np
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
DATA = "data/raw/content_refresh_anonymized.csv"

if IN_COLAB:
    # Clone my own repo (it ships the starter CSV); fall back to the FlyRank starter if needed.
    for url, folder in [("https://github.com/Ssaqlain09/flyrank-ml-internship", "flyrank-ml-internship"),
                        ("https://github.com/flyrank-bih/flyrank-ml-internship-starter", "flyrank-ml-internship-starter")]:
        if not os.path.isdir(folder):
            subprocess.run(["git", "clone", "--depth", "1", url, folder], check=True)
        if os.path.exists(os.path.join(folder, DATA)):
            os.chdir(folder)
            break
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

assert os.path.exists(DATA), "starter CSV not found - are you at the repo root?"
df = pd.read_csv(DATA)
y = (df["trend_direction"] == "down").astype(int)   # proxy label: ONLY for checking signals and scoring the rule, never a rule input

# Columns the rule must never touch: they define the label (or are IDs)
BANNED_INPUTS = ["trend_direction", "trend_pct", "impressions_last_30d", "impressions_prev_30d",
                 "clicks_last_30d", "clicks_prev_30d", "sessions_last_30d", "sessions_prev_30d",
                 "content_id", "client_id"]
RULE_INPUTS = ["impressions_90d", "avg_position", "ctr", "position_tier", "days_since_last_update"]

print(f"Rows: {len(df):,} | clients: {df['client_id'].nunique()} | base rate of the proxy label: {y.mean():.3f}")
print("Rule inputs:", RULE_INPUTS)

Rows: 30,000 | clients: 32 | base rate of the proxy label: 0.542
Rule inputs: ['impressions_90d', 'avg_position', 'ctr', 'position_tier', 'days_since_last_update']


### Signal check 1 — CTR vs position (the idea behind FlyRank's CTR-fix logic)

Question: among visible pages, do pages with a low CTR *for their position tier* decline more often?
Each page's CTR is divided by the median CTR of its position tier (a top-3 page and a page-4 page are
not compared directly). Position-tier medians of 0 (the `deep` tier) are undefined, so those pages drop out.

In [2]:
# Signal check 1: CTR vs position (the idea behind FlyRank's CTR-fix logic)
# CTR is compared with the median CTR of pages in the SAME position tier, on visible pages only.
visible = (df["impressions_90d"] >= 500) & (df["avg_position"] > 0)
tier_median = df.loc[visible].groupby("position_tier")["ctr"].median()
df["tier_median_ctr"] = df["position_tier"].map(tier_median).where(visible)
df["ctr_vs_tier"] = (df["ctr"] / df["tier_median_ctr"]).where(df["tier_median_ctr"] > 0)   # tier median 0 -> undefined

print("Median CTR per position tier (%, visible pages):", tier_median.round(2).to_dict())
b = pd.cut(df["ctr_vs_tier"], [0, 0.25, 0.5, 1, 2, np.inf], right=False,
           labels=["under 0.25x", "0.25-0.5x", "0.5-1x", "1-2x", "2x and up"])
sig1 = pd.DataFrame({"bucket": b, "declining": y}).dropna().groupby("bucket", observed=True)["declining"].agg(decline_rate="mean", n="size").round(3)
print()
print("CTR relative to its position tier's median  ->  share of pages declining")
print(sig1.to_string())
print(f"\nBase rate among visible pages: {y[visible].mean():.3f}")

Median CTR per position tier (%, visible pages): {'deep': 0.0, 'page_1': 0.24, 'page_3_5': 0.09, 'striking': 0.17, 'top_3': 0.2}

CTR relative to its position tier's median  ->  share of pages declining
             decline_rate     n
bucket                         
under 0.25x         0.682  2907
0.25-0.5x           0.677  1622
0.5-1x              0.624  3421
1-2x                0.584  4312
2x and up           0.518  4075

Base rate among visible pages: 0.596


### Signal check 2 — staleness (the idea behind FlyRank's refresh flags)

Question: do pages that were not updated for a long time decline more often? Because many pages share one
update age, I also compare *inside* each client, so the check is not just "which client is this".

In [3]:
# Signal check 2: staleness (the idea behind FlyRank's refresh flags)
b2 = pd.cut(df["days_since_last_update"], [0, 30, 90, 180, np.inf], right=False,
            labels=["0-29 days", "30-89", "90-179", "180+"])
sig2 = pd.DataFrame({"bucket": b2, "declining": y}).groupby("bucket", observed=True)["declining"].agg(decline_rate="mean", n="size").round(3)
print("Days since last update  ->  share of pages declining")
print(sig2.to_string())

# Is staleness a PAGE signal or a CLIENT signal? Many pages share one update age, so compare inside clients.
tmp = pd.DataFrame({"client_id": df["client_id"], "stale": (df["days_since_last_update"] >= 90).astype(int), "declining": y})
g = tmp.groupby(["client_id", "stale"])["declining"].agg(["mean", "size"]).unstack()
g.columns = ["fresh_rate", "stale_rate", "fresh_n", "stale_n"]
both = g.dropna()
both = both[(both["fresh_n"] >= 30) & (both["stale_n"] >= 30)].copy()
both["diff"] = both["stale_rate"] - both["fresh_rate"]
overall_gap = tmp[tmp.stale == 1]["declining"].mean() - tmp[tmp.stale == 0]["declining"].mean()
within_gap = np.average(both["diff"], weights=both["fresh_n"] + both["stale_n"])
print(f"\nStale (90+ days) minus fresh, all pages pooled:      {overall_gap:+.3f}")
print(f"Stale minus fresh, inside clients (>=30 pages each): {within_gap:+.3f}  "
      f"({(both['diff'] > 0).sum()} of {len(both)} clients show stale higher)")
print(f"Clients where every page has the same stale/fresh status: {(tmp.groupby('client_id')['stale'].mean().isin([0, 1])).mean():.0%}")

Days since last update  ->  share of pages declining
           decline_rate      n
bucket                        
0-29 days         0.511  20480
30-89             0.589    175
90-179            0.611   9171
180+              0.471    174

Stale (90+ days) minus fresh, all pages pooled:      +0.096
Stale minus fresh, inside clients (>=30 pages each): +0.053  (9 of 13 clients show stale higher)
Clients where every page has the same stale/fresh status: 34%


### Verdicts

| Signal | Verdict | Evidence (n printed above) |
|---|---|---|
| CTR relative to position tier (flag-linked) | **CONFIRMED** | The decline rate falls step by step as relative CTR rises: 0.682 (n=2,907) for under 0.25x, then 0.677, 0.624, 0.584, down to 0.518 (n=4,075) for 2x and up. The base rate among visible pages is 0.596. |
| Staleness, days since last update (flag-linked) | **MIXED** | Pages at 90+ days decline more than fresh ones (0.611 vs 0.511, a +0.096 gap), but inside clients the gap shrinks to +0.053, and only 9 of 13 clients show it. The 180+ bucket, which is where a product-style "stale" flag would sit, has just n=174 pages and points the other way (0.471), so it cannot be trusted. |

**What the verdicts changed in my rule.** CTR-vs-position becomes the main term (weight 0.8). Staleness is
kept only as a small bonus (weight 0.2) and at the **90-day** threshold, because that is where the evidence
is, not at 180 days where I have almost no pages. A rule built mainly on staleness would have leaned on a
signal that is partly a client effect.

**Careful words:** CONFIRMED means *associated with this proxy label in this dataset*. CTR and the 30-day
trend are measured in overlapping windows, so I do not claim that fixing a title would stop a decline.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Score, in plain terms: `100 × (0.8 × ctr_gap + 0.2 × stale_90) × demand_weight` for visible pages, and 0 for
everything else.

- `ctr_gap` runs from 0 (CTR at or above its tier median) to 1 (zero CTR).
- `stale_90` is 1 when the page was last updated 90 or more days ago.
- `demand_weight` grows with impressions and stops growing at 10,000 (so a page seen far more than that is not
  ranked ahead just for being huge).

All weights are set by hand — nothing is fitted. Ties are broken by more impressions first, then by
`content_id`, so the ranking is the same on every run.

In [4]:
# The rule (hand-set, nothing is fitted):
#   score = 100 * (0.8 * ctr_gap + 0.2 * stale_90) * demand_weight   for visible pages, 0 otherwise
W_GAP, W_STALE = 0.8, 0.2
df["ctr_gap"] = (1 - df["ctr_vs_tier"].clip(upper=1)).fillna(0.0)          # 0 = at/above tier median, 1 = zero CTR
df["stale_90"] = (df["days_since_last_update"] >= 90).astype(int)
df["demand_weight"] = np.minimum(np.log10(df["impressions_90d"].clip(lower=1)) / 4, 1)   # 10,000+ impressions = full weight

df["baseline_action_score"] = np.where(visible, 100 * (W_GAP * df["ctr_gap"] + W_STALE * df["stale_90"]) * df["demand_weight"], 0.0)
fires = visible & (df["ctr_gap"] >= 0.5)                                      # CTR under half of its tier's median
df["reason_code"] = np.where(fires, "low_ctr_for_position", "none")
df["action_label"] = np.where(fires, "review_title_snippet", "monitor")

queue = df.sort_values(["baseline_action_score", "impressions_90d", "content_id"], ascending=[False, False, True]).copy()
queue["rank"] = np.arange(1, len(queue) + 1)

out_cols = ["rank", "content_id", "client_id", "baseline_action_score", "reason_code", "action_label",
            "impressions_90d", "clicks_90d", "ctr", "tier_median_ctr", "avg_position", "position_tier", "days_since_last_update"]
os.makedirs("work/outputs", exist_ok=True)
queue[out_cols].round({"baseline_action_score": 2, "tier_median_ctr": 3}).to_csv("work/outputs/baseline_action_score.csv", index=False)

print(f"Wrote work/outputs/baseline_action_score.csv  ({len(queue):,} rows)")
print(f"Visible pages scored: {int(visible.sum()):,} | reason code fires on: {int(fires.sum()):,} | action = monitor on: {int((~fires).sum()):,}")
queue[out_cols].head(10).round(2)

Wrote work/outputs/baseline_action_score.csv  (30,000 rows)
Visible pages scored: 16,726 | reason code fires on: 4,713 | action = monitor on: 25,287


,rank,content_id,client_id,baseline_action_score,reason_code,action_label,impressions_90d,clicks_90d,ctr,tier_median_ctr,avg_position,position_tier,days_since_last_update
7445,1,content_c8e9d6ab9013,client_19581e27de,100.0,low_ctr_for_position,review_title_snippet,208678,0,0.0,0.24,9.7,page_1,104
8710,2,content_fb4bf6555c79,client_6208ef0f77,100.0,low_ctr_for_position,review_title_snippet,84093,3,0.0,0.09,45.6,page_3_5,104
26994,3,content_6e28a04c07a8,client_6208ef0f77,100.0,low_ctr_for_position,review_title_snippet,41226,2,0.0,0.09,34.3,page_3_5,104
25456,4,content_bc18d49d8f6b,client_6208ef0f77,100.0,low_ctr_for_position,review_title_snippet,32491,1,0.0,0.09,36.2,page_3_5,104
721,5,content_b21385c39124,client_6208ef0f77,100.0,low_ctr_for_position,review_title_snippet,30962,1,0.0,0.09,30.7,page_3_5,104
11521,6,content_75175d878762,client_6208ef0f77,100.0,low_ctr_for_position,review_title_snippet,25748,1,0.0,0.09,48.5,page_3_5,104
18509,7,content_095661034f9b,client_6208ef0f77,100.0,low_ctr_for_position,review_title_snippet,23513,1,0.0,0.09,39.4,page_3_5,104
25462,8,content_825a9788af8d,client_4e07408562,100.0,low_ctr_for_position,review_title_snippet,16786,0,0.0,0.24,5.6,page_1,104
9443,9,content_8ba781dafa55,client_8527a891e2,100.0,low_ctr_for_position,review_title_snippet,16156,0,0.0,0.24,9.0,page_1,104
20879,10,content_9597d14e7fa8,client_6208ef0f77,100.0,low_ctr_for_position,review_title_snippet,10932,0,0.0,0.09,27.0,page_3_5,104


## 3. Top-10 review

*For each of the top ten: the action, why it is there, and what would make it wrong.*

In [5]:
top10 = queue.head(10)
client_counts = top10["client_id"].value_counts()
same_age_share = df.groupby("client_id")["days_since_last_update"].agg(lambda s: s.value_counts().iloc[0] / len(s))

def review_line(idx, r):
    why = (f"{r['impressions_90d']:,} impressions at position {r['avg_position']:.0f}, CTR {r['ctr']:.2f}% "
           f"vs tier median {r['tier_median_ctr']:.2f}%, last updated {r['days_since_last_update']} days ago")
    wrong = []
    if r["position_tier"] == "page_1" and r["clicks_90d"] <= 2 and r["impressions_90d"] >= 10_000:
        wrong.append("near-zero clicks on a page-1 result with this many impressions is unusual, so check for a tracking or measurement problem before rewriting the title")
    if r["avg_position"] > 30:
        wrong.append(f"at position {r['avg_position']:.0f} few people see the result, so a tiny CTR may simply be normal")
    if client_counts[r["client_id"]] >= 4:
        wrong.append(f"{client_counts[r['client_id']]} of the top 10 come from one client, so the cause may be site-wide, not this page")
    if r["days_since_last_update"] >= 90 and same_age_share[r["client_id"]] >= 0.5:
        wrong.append(f"{same_age_share[r['client_id']]:.0%} of this client's pages share one update age, so it may be a bulk date, not a real staleness signal")
    if r["clicks_90d"] == 0 and df.loc[idx, "sessions_90d"] >= 20:
        wrong.append(f"0 clicks but {int(df.loc[idx, 'sessions_90d'])} sessions, so the click data may be incomplete")
    return f"#{r['rank']}  {r['action_label']}  [{r['reason_code']}]\n   why: {why}\n   wrong if: " + "; ".join(wrong) + "."

for idx, r in top10.iterrows():
    print(review_line(idx, r)); print()

#1  review_title_snippet  [low_ctr_for_position]
   why: 208,678 impressions at position 10, CTR 0.00% vs tier median 0.24%, last updated 104 days ago
   wrong if: near-zero clicks on a page-1 result with this many impressions is unusual, so check for a tracking or measurement problem before rewriting the title.

#2  review_title_snippet  [low_ctr_for_position]
   why: 84,093 impressions at position 46, CTR 0.00% vs tier median 0.09%, last updated 104 days ago
   wrong if: at position 46 few people see the result, so a tiny CTR may simply be normal; 7 of the top 10 come from one client, so the cause may be site-wide, not this page; 81% of this client's pages share one update age, so it may be a bulk date, not a real staleness signal.

#3  review_title_snippet  [low_ctr_for_position]
   why: 41,226 impressions at position 34, CTR 0.00% vs tier median 0.09%, last updated 104 days ago
   wrong if: at position 34 few people see the result, so a tiny CTR may simply be normal; 7 of the top 1

**My reading of the top ten, as a skeptic:**

- All ten pages hit the maximum score of 100 (zero clicks, stale, 10,000+ impressions), so their order among
  themselves is just the impressions tie-break. The score cannot tell #1 from #10.
- **7 of the 10 come from a single client**, and 81% of that client's pages share the same update age (104
  days). That looks more like a client-level pattern (one template, one bulk update, or one tracking setup)
  than ten separate page problems. The honest first action for that client may be one site-level check, not
  seven title rewrites.
- Seven picks sit at positions 27–48, where a tiny CTR is expected because few people scroll that far.
  The other three sit on page 1 (positions 6–10) with near-zero clicks on 16,000–208,000 impressions; that
  is unusual enough that I would check the data (tracking, impression counting) before touching the title.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

I use the proxy label here only to look back at the picks, never as a rule input. Then I re-score the rule
without the client that dominates the top 100, to see how much of the result is one client.

In [6]:
# (a) Which top-10 picks look weak? The proxy label is used ONLY to look back at the picks.
top10_eval = queue.head(10).assign(proxy_declining=y.loc[queue.head(10).index].values)
print("Top-10 picks where the proxy label says NOT declining:")
print(top10_eval.loc[top10_eval["proxy_declining"] == 0, ["rank", "content_id", "impressions_90d", "ctr", "avg_position", "trend_direction"]].to_string(index=False))
print(f"Top-10 picks with the label saying declining: {int(top10_eval['proxy_declining'].sum())} of 10")
print(f"Score ties inside the top 10 (score = {queue['baseline_action_score'].iloc[0]:.0f}): {int((queue.head(10)['baseline_action_score'] == queue['baseline_action_score'].iloc[0]).sum())} of 10 share the max score")

# (b) Precision@K against two base rates (fixed rule, nothing fitted; thresholds were chosen after looking at the tables above)
def precision_at_k(q, k):
    return float(y.loc[q.head(k).index].mean())
pk = {k: round(precision_at_k(queue, k), 3) for k in (10, 20, 50, 100)}
clients_in_top = {k: int(queue.head(k)["client_id"].nunique()) for k in (10, 20, 50, 100)}
print(f"\nPrecision@K of the rule: {pk}")
print(f"Base rate, all pages: {y.mean():.3f} | base rate, visible pages only: {y[visible].mean():.3f}")
print(f"Distinct clients inside the top K: {clients_in_top}   (out of {df['client_id'].nunique()} clients)")

# (c) Is it one client doing the work? Drop the client that dominates the top 100 and re-score.
big = queue.head(100)["client_id"].value_counts().index[0]
share100 = (queue.head(100)["client_id"] == big).mean()
rest = queue[queue["client_id"] != big]
pk_rest = {k: round(precision_at_k(rest, k), 3) for k in (10, 50, 100)}
print(f"\nLargest client in the top 100 holds {share100:.0%} of it; that client's overall decline rate is {y[df['client_id'] == big].mean():.3f}")
print(f"Precision@K after removing that client: {pk_rest}  (base rate without it: {y[df['client_id'] != big].mean():.3f}, visible: {y[visible & (df['client_id'] != big)].mean():.3f})")

# (d) Leakage check: no rule input may be a label-defining column or an ID
assert set(RULE_INPUTS).isdisjoint(BANNED_INPUTS), "a banned column is used by the rule"
print("\nLeakage check passed: rule inputs", RULE_INPUTS, "contain no future-window or label-derived column.")

# (e) Receipts for the report (small JSON; the CSV itself stays out of git)
metrics = {"rule": "100*(0.8*ctr_gap + 0.2*stale_90)*demand_weight on visible pages; reason code low_ctr_for_position when CTR < 0.5x tier median",
           "visible_definition": "impressions_90d >= 500 and avg_position > 0",
           "weights": {"ctr_gap": W_GAP, "stale_90": W_STALE}, "rows": int(len(df)), "visible_rows": int(visible.sum()),
           "reason_code_fires": int(fires.sum()), "base_rate_all": round(float(y.mean()), 3),
           "base_rate_visible": round(float(y[visible].mean()), 3), "precision_at_k": pk,
           "precision_at_k_without_largest_client": pk_rest,
           "distinct_clients_in_top_k": clients_in_top, "label": "proxy: trend_direction == down (evaluation only)",
           "validation": "in-sample: fixed rule, thresholds chosen after viewing bucket tables on the same data"}
with open("work/outputs/baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("Wrote work/outputs/baseline_metrics.json")

Top-10 picks where the proxy label says NOT declining:
 rank           content_id  impressions_90d  ctr  avg_position trend_direction
    4 content_bc18d49d8f6b            32491  0.0          36.2          stable
Top-10 picks with the label saying declining: 9 of 10
Score ties inside the top 10 (score = 100): 10 of 10 share the max score

Precision@K of the rule: {10: 0.9, 20: 0.95, 50: 0.76, 100: 0.77}
Base rate, all pages: 0.542 | base rate, visible pages only: 0.596
Distinct clients inside the top K: {10: 4, 20: 5, 50: 7, 100: 8}   (out of 32 clients)

Largest client in the top 100 holds 57% of it; that client's overall decline rate is 0.644
Precision@K after removing that client: {10: 0.8, 50: 0.68, 100: 0.72}  (base rate without it: 0.528, visible: 0.589)

Leakage check passed: rule inputs ['impressions_90d', 'avg_position', 'ctr', 'position_tier', 'days_since_last_update'] contain no future-window or label-derived column.
Wrote work/outputs/baseline_metrics.json


**What the checks show.**

- **One weak pick in the top ten:** rank 4 is labelled `stable`, not declining, so 9 of the top 10 are
  declining (Precision@10 = 0.90). Its profile (position 36, zero clicks) is the same as its neighbours, so
  the rule cannot see why it differs.
- **Precision@K of the rule:** 0.90 at 10, 0.95 at 20, 0.76 at 50 and 0.77 at 100, against a base rate of 0.542
  for all pages and 0.596 for visible pages. The top of the list beats both base rates.
- **But one client carries a lot of it.** The largest client in the top 100 holds 57% of it, and that client
  declines 64.4% of the time overall. Without that client the rule scores 0.80 at 10, 0.68 at 50 and 0.72 at
  100, against base rates of 0.528 (all) and 0.589 (visible). It still beats the base rate, by less.
- **Leakage check:** an assertion confirms none of the five rule inputs is a label-defining or ID column. No
  future-window column is used.

**Limits, in careful words.** This is a fixed rule checked on the same 30,000 rows I used to choose its
thresholds, so the numbers are optimistic and directional, not proof. The label is a proxy from one window.
This baseline is **frozen**: my Week-5 model has to beat Precision@50 = 0.76 (0.68 without the largest client)
on the same data slice, or it has not earned its complexity.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.